In [ ]:
import numpy as np
import pandas as pd
import pickle
import networkx as nx
from pathlib import Path
from tqdm import tqdm
from ssl_neuron.utils import neighbors_to_adjacency
from data_utils import connect_graph

In [4]:
import sys
import numpy.core

sys.modules["numpy._core"] = numpy.core
sys.modules["numpy._core.multiarray"] = numpy.core.multiarray
sys.modules["numpy._core.numeric"] = numpy.core.numeric

In [ ]:
CSV_PATH = Path(
    "/gpfs01/berens/data/data/Eyewire2/morphologies-ew2/csv/main_all_cells/"
    "Eyewire II Proofread Cells Main List - All Cells 2026-11-03.csv"
)

PICKLE_PATH = Path(
    "/gpfs01/berens/data/data/Eyewire2/database-exports/clf_rgcs_and_acs/"
    "df_rgcs_and_acs_2026-11-02-19h.pickle"
)

In [ ]:
df_labels = pd.read_csv(CSV_PATH)

df_features = pd.read_pickle(PICKLE_PATH)

print("CSV:", df_labels.shape)
print("Pickle:", df_features.shape)

print(df_labels.columns.tolist())
print(df_features.columns.tolist())

/tmp/ipykernel_1808130/1032532387.py:3: DtypeWarning: Columns (19,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  df_labels = pd.read_csv(CSV_PATH)


CSV: (30139, 62)
Pickle: (26159, 417)
['Index', 'Nuc Coords', 'Latest NucID', 'Soma Coords', 'Latest SegID', 'Latest SegID Update Date: 2026-10-30', 'Proofreader Cell Type Guess', 'Proofreader 1', 'Proofreader 2', 'Status', 'AIS annotated?', 'Proofread Link', 'Proofread NucID', 'Proofread SegID', 'Updater', 'Date Updated', 'Proofreading Notes', 'Exclude (empty will be treated as Include)', 'Include in Resource Paper', 'Directionality', 'Mesh Status', 'Mesh Status (machine)', 'Cell Class', 'Cell Class (machine)', 'Confidence Cell Class (machine)', 'Reliability Human Label', 'Cell Typer Name', 'Cell Type', 'Cell Type (machine)', 'Confidence Cell Type (machine)', 'Other Type', 'EW auto layer type', 'L0 weight', 'L1 weight', 'L2 weight', 'L3 weight', 'L4 weight', 'L5 weight', 'L6 weight', 'L7 weight', 'L8 weight', 'L9 weight', 'L10(X) weight', 'L11(Y) weight', 'L12(Z) weight', 'EW2 type', 'EW2 type (machine)', 'Confidence EW2 type (machine)', 'EyeWire1 type (for RGCs) human label', 'Reliab

In [ ]:
df = df_features.copy()

# Keep only RGCs
df = df[
    (df["cellclass"] == "RGC")
].copy()

print(f"RGCs with an NPZ path: {len(df)}")

RGCs with an NPZ path: 3831


In [72]:
df[
    (df["cellclass"] == "RGC") &
    (df["celltype_conf"] == "High")
].shape

(2050, 418)

In [76]:
print(df_features.columns.to_list())

['skel_file', 'n_nodes', 'n_edges', 'post_has_soma', 'post_has_axon', 'mesh_file', 'soma_x_um', 'soma_y_um', 'soma_z_um', 'soma_rad_um', 'index', 'link', 'status', 'status_alt', 'req_review', 'ais', 'cellclass', 'cellclass_alt', 'mesh_status', 'mesh_status_alt', 'celltype', 'celltype_conf', 'ew2type', 'ew2type_conf', 'ew1type', 'ew1type_conf', 'celltype_alt', 'soma_annot_x_vox', 'soma_annot_y_vox', 'soma_annot_z_vox', 'nuc_annot_x_vox', 'nuc_annot_y_vox', 'nuc_annot_z_vox', 'nuc', 'rows', 'n_nucleus', 'n_soma', 'n_true_end', 'n_cant_fix', 'n_hits_edge', 'n_ais', 'ais_x_um', 'ais_y_um', 'ais_z_um', 'polar_dens_id_1_x', 'total_length_1', 'polar_dens_r_1', 'polar_dens_t_1', 'polar_dens_z_1', 'polar_dens_rt_res_1', 'polar_dens_rz_res_1', 'polar_dens_tz_res_1', 'var_expl_z1d_1', 'var_expl_1d_1', 'var_expl_2d_1', 'polar_center_x_um_1', 'polar_center_y_um_1', 'polar_dens_id_2', 'total_length_2', 'polar_dens_r_2', 'polar_dens_t_2', 'polar_dens_z_2', 'polar_dens_rt_res_2', 'polar_dens_rz_res_2'

assuming axon is node type 2

- drop axons <br>
- connectivity check after axon removal


In [ ]:
OUT_DIR = Path.cwd()
SKELETON_DIR = OUT_DIR / "skeletons"
SKELETON_DIR.mkdir(exist_ok=True)
MIN_NODES = 1000

def remove_axon(neighbors, features, soma_id):
    axon_mask = features[:, 5] == 1
    keep = np.where(~axon_mask)[0]

    features = features[keep]
    old2new = {old: new for new, old in enumerate(keep)}

    neighbors = {
        old2new[k]: {old2new[n] for n in v if n in old2new}
        for k, v in neighbors.items() if k in old2new
    }

    return neighbors, features, old2new[soma_id]


def type_to_onehot(types):
    onehot = np.zeros((len(types), 4), dtype=np.float32)
    onehot[np.isin(types, [1, -1]), 0] = 1  # soma
    onehot[types == 2, 1] = 1              # axon
    onehot[types == 3, 2] = 1              # dendrite
    onehot[types >= 4, 3] = 1              # apical dendrite
    return onehot


def remap_neighbors(neighbors):
    old2new = {old: new for new, old in enumerate(sorted(neighbors))}
    neighbors = {
        old2new[k]: {old2new[n] for n in v if n in old2new}
        for k, v in neighbors.items()
    }
    return neighbors, old2new


def preprocess_npz(npz_file):
    with np.load(npz_file) as data:
        nodes = data["nodes"].astype(np.float32)
        edges = data["edges"].astype(int)
        types = data["ntype"]
        radii = data["r_median"].astype(np.float32)

    onehot = type_to_onehot(types)
    features = np.column_stack([nodes, radii, onehot])

    soma_ids = np.where(onehot[:, 0] == 1)[0]
    if len(soma_ids) != 1:
        raise ValueError(f"Expected one soma, found {len(soma_ids)}")

    neighbors = {i: set() for i in range(len(nodes))}
    for i, j in edges:
        neighbors[i].add(j)
        neighbors[j].add(i)

    # Move soma to node 0
    soma_id = int(soma_ids[0])
    if soma_id != 0:
        order = np.arange(len(features))
        order[[0, soma_id]] = order[[soma_id, 0]]
        old2new = {old: new for new, old in enumerate(order)}
        features = features[order]
        neighbors = {
            old2new[k]: {old2new[n] for n in v}
            for k, v in neighbors.items()
        }

    n_raw = len(features)
    n_axon = int(features[:, 5].sum())

    # Connect graph before axon removal
    adj = neighbors_to_adjacency(neighbors, range(len(neighbors)))
    adj, neighbors = connect_graph(adj, neighbors, features)

    # Remove axon and remap indices
    neighbors, features, soma_id = remove_axon(neighbors, features, 0)

    # Connect graph after axon removal
    adj = neighbors_to_adjacency(neighbors, range(len(neighbors)))
    adj, neighbors = connect_graph(adj, neighbors, features)

    if len(features) < MIN_NODES:
        raise ValueError(f"Only {len(features)} nodes after axon removal")

    # Center x/y only; preserve z
    soma_xyz = features[soma_id, :3].copy()
    features[:, :2] -= soma_xyz[:2]

    info = {
        "n_nodes_raw": n_raw,
        "n_nodes": len(features),
        "n_axon_nodes": n_axon,
        "n_stitched_edges": adj.sum() // 2 - (len(edges) // 2)
    }

    return features[:, :3].astype(np.float32), neighbors, soma_xyz, info


# Process cells
df = df[df["skel_file"].notna()].copy()
df["npz_file"] = df["skel_file"].astype(str)
df["cell_id"] = df["npz_file"].apply(lambda x: Path(x).stem)

metadata, failed = [], []

for _, row in tqdm(df.iterrows(), total=len(df)):
    cell_id = row["cell_id"]
    npz_file = Path(row["npz_file"])

    try:
        features, neighbors, soma_xyz, info = preprocess_npz(npz_file)

        cell_dir = SKELETON_DIR / cell_id
        cell_dir.mkdir(exist_ok=True)

        np.save(cell_dir / "features.npy", features)
        with open(cell_dir / "neighbors.pkl", "wb") as f:
            pickle.dump(neighbors, f)

        metadata.append({
            "cell_id": cell_id,
            "soma_x": soma_xyz[0],
            "soma_y": soma_xyz[1],
            "soma_z": soma_xyz[2],
            **info,
            "cellclass_final": row.get("cellclass", np.nan),
            "valid_cellclass_final": row.get("valid_cellclass", np.nan),
            "celltype_final": row.get("celltype", np.nan),
            "valid_celltype_final": row.get("valid_celltype", np.nan),
            "celltype_conf": row.get("celltype_conf", np.nan),
            "valid_status": row.get("valid_status", np.nan),
            "npz_file": str(npz_file)
        })

    except Exception as e:
        print('fail',  str(e))
        failed.append((cell_id, str(e)))

pd.DataFrame(metadata).to_csv(OUT_DIR / "cell_meta.csv", index=False)

print(f"Processed: {len(metadata)}")
print(f"Failed: {len(failed)}")
if failed:
    print(failed[:10])

 35%|███▌      | 1356/3831 [58:10<1:30:21,  2.19s/it]

fail Only 661 nodes after axon removal


100%|██████████| 3831/3831 [2:43:13<00:00,  2.56s/it]  

Processed: 3830
Failed: 1
[('720575940560435035', 'Only 661 nodes after axon removal')]
